## <h1><center>POYO Single-Session Decoding on DANDI:000688</center></h1>

<center><b>CS 4782 Final Project Reproduction Notebook</b></center>

&nbsp;

---

**Goal:** Reproduce a single-session POYO velocity decoding experiment from *A Unified, Scalable Framework for Neural Population Decoding* using one automatically selected session from DANDI `000688`, then compare POYO against a Wiener filter, MLP, and GRU.

**Success means:** the notebook runs end-to-end from a fresh Colab runtime, inspects the NWB structure before using fields, trains all four models, reports `R^2` on held-out data, renders the required figures, and saves artifacts under `./artifacts/`.


# Part 0: Requirements Checklist

This notebook was grounded against the local course materials before implementation:

- `DL Final Project proposal.pdf`: target reproduction is single-session POYO velocity decoding with Wiener / MLP / GRU baselines; the proposal cites a stretch target near `R^2 ≈ 0.97`.
- `CS 4782 Final Project Instructions.pdf`: emphasize clear methodology, visual results, discussion of discrepancies, and reproducibility.
- `POYO1.pdf`: use a `1s` context window, `R^2` as the primary metric, and trial-based `20%` test / `10%` validation splits when trials exist.
- `Assignment 2` and `Assignment 3`: follow the same narrative style with setup first, clear sectioning, and visuals throughout.

Implementation checklist satisfied here:

- Auto-select the smallest valid published session from DANDI `000688`.
- Inspect NWB structure before touching any field names.
- Train and evaluate Wiener, MLP, GRU, and POYO end-to-end.
- Save figures plus `metrics.json` under `./artifacts/`.
- Compare results to the paper and explain any mismatch.

Important note:

- Because this notebook auto-selects the smallest valid session, it may land on a random-target session rather than a center-out session. The proposal's `~0.97` target is more consistent with easier center-out conditions, while the paper's Section 3.2 reports a lower average for random-target sessions. We will call that out explicitly in the final discussion.


In [ ]:
# Top-of-notebook configuration cell.

FAST_DEV_RUN = True
RANDOM_SEED = 42

CONTEXT_SEC = 1.0
BIN_SIZE_SEC = 0.01
TARGET_STRIDE_SEC = 0.10 if FAST_DEV_RUN else 0.05

MAX_CANDIDATES_TO_CHECK = 5
MAX_EPOCHS_MLP = 3 if FAST_DEV_RUN else 20
MAX_EPOCHS_GRU = 3 if FAST_DEV_RUN else 20
MAX_EPOCHS_POYO = 2 if FAST_DEV_RUN else 10

MLP_BATCH_SIZE = 256
GRU_BATCH_SIZE = 128
POYO_BATCH_SIZE = 48 if FAST_DEV_RUN else 64

POYO_LR = 1e-3
POYO_WEIGHT_DECAY = 1e-4

FAST_DEV_MAX_SAMPLES = {"train": 1200, "val": 300, "test": 400}


# Part 1: Environment Setup

In [ ]:
%pip -q install dandi pynwb temporaldata matplotlib pandas scikit-learn nwbwidgets \
    'torchmetrics>=1.6.0' einops==0.6.1 hydra-core==1.3.2 torchtyping==0.1.5 rich pytorch_brain

from pathlib import Path
import subprocess

Path("third_party").mkdir(exist_ok=True)
if not Path("third_party/torch_brain").exists():
    !git clone --depth 1 https://github.com/neuro-galaxy/torch_brain.git third_party/torch_brain

clone_commit = subprocess.check_output(
    ["git", "-C", "third_party/torch_brain", "rev-parse", "HEAD"],
    text=True,
).strip()
print("torch_brain clone commit:", clone_commit)


In [ ]:
import os
import platform
import numpy as np
import pandas as pd
import torch
import torch_brain

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Python:", platform.python_version())
print("Platform:", platform.platform())
print("Torch:", torch.__version__)
print("torch_brain:", getattr(torch_brain, "__version__", "unknown"))
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

torch.use_deterministic_algorithms(False)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_SEED)


In [ ]:
import inspect
import subprocess
from torch_brain.models import POYO
from torch_brain.registry import MODALITY_REGISTRY

print("Verified POYO signature:")
print(inspect.signature(POYO))
print("\nVerified cursor readout spec:")
print(MODALITY_REGISTRY["cursor_velocity_2d"])
print("\nReference files in the official clone:")
print(
    subprocess.check_output(
        ["bash", "-lc", "find third_party/torch_brain/examples/poyo -maxdepth 2 -type f | sort"],
        text=True,
    )
)


# Part 2: Shared Support Code

## 2.1 Imports

In [ ]:
from __future__ import annotations

import copy
import json
import math
import random
import sys
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, Iterable, List, Tuple

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch_brain
from dandi.dandiapi import DandiAPIClient
from pynwb import NWBHDF5IO
from sklearn.linear_model import Ridge
from sklearn.metrics import r2_score
from temporaldata import ArrayDict, Interval, IrregularTimeSeries
from torch.utils.data import DataLoader, Dataset, TensorDataset

from torch_brain.data import collate as tb_collate
from torch_brain.models import POYO
from torch_brain.optim import SparseLamb
from torch_brain.registry import MODALITY_REGISTRY

## 2.2 Paths and Configuration

In [ ]:
PROJECT_ROOT = Path.cwd()
ARTIFACTS_DIR = PROJECT_ROOT / "artifacts"
TMP_DIR = PROJECT_ROOT / "tmp" / "dandi_inspect"
ARTIFACTS_DIR.mkdir(exist_ok=True, parents=True)
TMP_DIR.mkdir(exist_ok=True, parents=True)


FAST_DEV_RUN = globals().get("FAST_DEV_RUN", True)
RANDOM_SEED = globals().get("RANDOM_SEED", 42)
CONTEXT_SEC = globals().get("CONTEXT_SEC", 1.0)
BIN_SIZE_SEC = globals().get("BIN_SIZE_SEC", 0.01)
TARGET_STRIDE_SEC = globals().get("TARGET_STRIDE_SEC", 0.10 if FAST_DEV_RUN else 0.05)
MAX_CANDIDATES_TO_CHECK = globals().get("MAX_CANDIDATES_TO_CHECK", 5)
MAX_EPOCHS_MLP = globals().get("MAX_EPOCHS_MLP", 3 if FAST_DEV_RUN else 20)
MAX_EPOCHS_GRU = globals().get("MAX_EPOCHS_GRU", 3 if FAST_DEV_RUN else 20)
MAX_EPOCHS_POYO = globals().get("MAX_EPOCHS_POYO", 2 if FAST_DEV_RUN else 10)
MLP_BATCH_SIZE = globals().get("MLP_BATCH_SIZE", 256)
GRU_BATCH_SIZE = globals().get("GRU_BATCH_SIZE", 128)
POYO_BATCH_SIZE = globals().get("POYO_BATCH_SIZE", 48 if FAST_DEV_RUN else 64)
POYO_LR = globals().get("POYO_LR", 1e-3)
POYO_WEIGHT_DECAY = globals().get("POYO_WEIGHT_DECAY", 1e-4)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
FAST_DEV_MAX_SAMPLES = globals().get("FAST_DEV_MAX_SAMPLES", {"train": 1200, "val": 300, "test": 400})

## 2.3 Data Classes

In [ ]:
@dataclass
class SimpleSessionDescription:
    """
    INPUT:
    id : unique session identifier (str)

    OUTPUT:
    None (dataclass container)
    """
    id: str


class MiniData:
    def __init__(self, **kwargs):
        """
        INPUT:
        kwargs : arbitrary keyword arguments, each becomes an instance attribute (dict)

        OUTPUT:
        None (constructor)
        """
        # pseudocode: copy every provided keyword argument onto this lightweight container object
        self.__dict__.update(kwargs)

    def get_nested_attribute(self, dotted_key: str):
        """
        INPUT:
        dotted_key : dot-separated attribute path e.g. "config.readout" (str)

        OUTPUT:
        obj        : the value found by traversing the attribute chain (any)
        """
        # pseudocode: start the lookup at the MiniData object itself
        obj = self
        # pseudocode: walk one attribute name at a time through the dotted path
        for part in dotted_key.split("."):
            # pseudocode: advance to the next nested attribute before continuing the traversal
            obj = getattr(obj, part)
        # pseudocode: return the final value reached by the chained attribute lookup
        return obj


@dataclass
class SelectedAsset:
    """
    INPUT:
    dandiset_id         : DANDI archive identifier (str)
    version_id          : published dandiset version identifier (str)
    asset_id            : unique asset identifier within DANDI (str)
    path                : remote asset path inside the dandiset (str)
    size_bytes          : remote file size in bytes (int)
    local_path          : downloaded local filesystem path (Path)
    subject_id          : subject identifier if present in NWB metadata (str | None)
    session_description : free-text NWB session description (str)
    duration_sec        : duration of the behavioral recording in seconds (float)
    units_count         : number of recorded neural units (int)
    trials_count        : number of trial rows in the NWB file (int)

    OUTPUT:
    None (dataclass container)
    """
    dandiset_id: str
    version_id: str
    asset_id: str
    path: str
    size_bytes: int
    local_path: Path
    subject_id: str | None
    session_description: str
    duration_sec: float
    units_count: int
    trials_count: int

## 2.4 Seed and Serialisation Helpers

In [ ]:
def set_seed(seed: int) -> None:
    """
    INPUT:
    seed : integer seed for all random number generators (int)

    OUTPUT:
    None
    """
    # pseudocode: seed Python's built-in random module
    random.seed(seed)
    # pseudocode: seed NumPy's legacy random number generator
    np.random.seed(seed)
    # pseudocode: seed PyTorch's CPU random number generator
    torch.manual_seed(seed)
    # pseudocode: if CUDA is available, also make GPU-side randomness reproducible
    if torch.cuda.is_available():
        # pseudocode: broadcast the same seed to every visible CUDA device
        torch.cuda.manual_seed_all(seed)


def json_default(obj):
    """
    INPUT:
    obj : a Python object the default JSON encoder cannot handle (any)

    OUTPUT:
    serializable : a JSON-serializable version of obj (int, float, list, or str)
    """
    # pseudocode: convert NumPy scalar wrappers into plain Python numbers
    if isinstance(obj, (np.floating, np.integer)):
        # pseudocode: extract the underlying Python scalar value
        return obj.item()
    # pseudocode: convert NumPy arrays into nested Python lists
    if isinstance(obj, np.ndarray):
        # pseudocode: serialize the array contents as a list
        return obj.tolist()
    # pseudocode: serialize filesystem paths as their string representation
    if isinstance(obj, Path):
        # pseudocode: cast the Path object into text for JSON output
        return str(obj)
    # pseudocode: fail loudly for unsupported objects so serialization bugs are visible
    raise TypeError(f"Unsupported type: {type(obj)}")

## 2.5 DANDI Asset Discovery

In [ ]:
def latest_published_dandiset(dandiset_id: str = "000688"):
    """
    INPUT:
    dandiset_id : DANDI archive identifier (str)

    OUTPUT:
    ds : dandiset handle for the latest published version (RemoteDandiset)
    """
    # pseudocode: open a DANDI API client session for this metadata lookup
    with DandiAPIClient() as client:
        # pseudocode: fetch the latest published dandiset object for the requested identifier
        ds = client.get_dandiset(dandiset_id)
        # pseudocode: return the remote dandiset handle so later code can inspect its assets
        return ds


def download_smallest_valid_asset(
    dandiset_id: str = "000688",
    candidate_limit: int = MAX_CANDIDATES_TO_CHECK,
) -> SelectedAsset:
    """
    INPUT:
    dandiset_id     : DANDI archive identifier (str)
    candidate_limit : max NWB files to inspect (int)

    OUTPUT:
    selected : metadata and local path for the smallest valid NWB file (SelectedAsset)
    """
    # pseudocode: fetch the newest published dandiset before scanning its files
    ds = latest_published_dandiset(dandiset_id)
    # pseudocode: collect candidate NWB assets together with their file sizes
    candidate_rows: List[Tuple[int, object]] = []
    # pseudocode: iterate through the dandiset assets in path order for deterministic selection
    for asset in ds.get_assets(order="path"):
        # pseudocode: keep only NWB files because those contain the neural recording session data
        if asset.path.endswith(".nwb"):
            # pseudocode: store each NWB asset alongside its size for later ranking
            candidate_rows.append((asset.size, asset))
    # pseudocode: rank the candidate NWB files from smallest to largest
    candidate_rows.sort(key=lambda x: x[0])

    # pseudocode: inspect only the smallest candidate files until one passes every validation check
    for size_bytes, asset in candidate_rows[:candidate_limit]:
        # pseudocode: map the remote asset name to a local temporary download path
        local_path = TMP_DIR / Path(asset.path).name
        # pseudocode: download the candidate if it is missing locally or previously truncated
        if not local_path.exists() or local_path.stat().st_size == 0:
            # pseudocode: report which NWB candidate is being downloaded for inspection
            print(f"Downloading candidate: {asset.path} ({size_bytes / 1e6:.2f} MB)")
            # pseudocode: fetch the asset bytes from DANDI into the temporary directory
            asset.download(local_path)

        # pseudocode: try to open the downloaded NWB file and verify it contains the needed signals
        try:
            # pseudocode: load the NWB file with namespaces so custom interfaces deserialize correctly
            with NWBHDF5IO(str(local_path), "r", load_namespaces=True) as io:
                # pseudocode: parse the NWB file into an in-memory object model
                nwb = io.read()
                # pseudocode: pull the units table if it exists so we can count neural channels
                units = getattr(nwb, "units", None)
                # pseudocode: count the number of units, defaulting to zero when the table is absent
                units_count = 0 if units is None else len(units.id[:])
                # pseudocode: pull the trials table if it exists so we can count behavioral trials
                trials = getattr(nwb, "trials", None)
                # pseudocode: count the number of trials, defaulting to zero when no trial table exists
                trials_count = 0 if trials is None else len(trials.id[:])
                # pseudocode: look for the behavior processing module that should hold cursor signals
                behavior = nwb.processing.get("behavior", None)
                # pseudocode: skip files that do not contain any behavior module at all
                if behavior is None:
                    continue
                # pseudocode: look for the velocity interface inside the behavior module
                velocity_iface = behavior.data_interfaces.get("Velocity", None)
                # pseudocode: look for the position interface inside the behavior module
                position_iface = behavior.data_interfaces.get("Position", None)
                # pseudocode: verify that a cursor velocity time series is available
                has_velocity = (
                    velocity_iface is not None
                    and hasattr(velocity_iface, "time_series")
                    and "cursor_vel" in velocity_iface.time_series
                )
                # pseudocode: verify that a cursor position spatial series is available
                has_position = (
                    position_iface is not None
                    and hasattr(position_iface, "spatial_series")
                    and "cursor_pos" in position_iface.spatial_series
                )
                # pseudocode: reject files without enough units, trials, or cursor trajectory information
                if not (units_count > 0 and trials_count >= 20 and (has_velocity or has_position)):
                    continue
                # pseudocode: measure the recording duration from velocity timestamps when velocity exists
                if has_velocity:
                    # pseudocode: load the cursor velocity time series
                    vel = velocity_iface.time_series["cursor_vel"]
                    # pseudocode: copy the velocity timestamps into a NumPy array
                    ts = np.asarray(vel.timestamps[:])
                    # pseudocode: compute duration from the first and last timestamps
                    duration_sec = float(ts[-1] - ts[0]) if len(ts) else 0.0
                # pseudocode: otherwise estimate duration from the position timestamps
                else:
                    # pseudocode: load the cursor position spatial series
                    pos = position_iface.spatial_series["cursor_pos"]
                    # pseudocode: copy the position timestamps into a NumPy array
                    ts = np.asarray(pos.timestamps[:])
                    # pseudocode: compute duration from the first and last timestamps
                    duration_sec = float(ts[-1] - ts[0]) if len(ts) else 0.0
                # pseudocode: skip recordings that are too short to support the project workflow
                if duration_sec < 120.0:
                    continue
                # pseudocode: package the validated asset metadata and return the winning candidate
                return SelectedAsset(
                    dandiset_id=dandiset_id,
                    version_id=ds.version_id,
                    asset_id=asset.identifier,
                    path=asset.path,
                    size_bytes=size_bytes,
                    local_path=local_path,
                    subject_id=getattr(getattr(nwb, "subject", None), "subject_id", None),
                    session_description=nwb.session_description,
                    duration_sec=duration_sec,
                    units_count=units_count,
                    trials_count=trials_count,
                )
        # pseudocode: if opening or validating this file fails, log it and try the next candidate
        except Exception as exc:
            # pseudocode: show the reason this asset was rejected
            print(f"Skipping {asset.path}: {exc!r}")
            # pseudocode: continue searching through the remaining candidate files
            continue

    # pseudocode: raise an error when no candidate NWB asset satisfies the quality requirements
    raise RuntimeError("Could not find a valid asset among the inspected candidates.")

## 2.6 NWB Loading and Inspection

In [ ]:
def inspect_nwb_file(local_path: Path) -> Dict:
    """
    INPUT:
    local_path : path to downloaded NWB file (Path)

    OUTPUT:
    structure : dict summarizing NWB top-level groups (dict)
    """
    # pseudocode: open the NWB file so we can inspect its top-level organization
    with NWBHDF5IO(str(local_path), "r", load_namespaces=True) as io:
        # pseudocode: deserialize the NWB file into a Python object hierarchy
        nwb = io.read()
        # pseudocode: summarize the key top-level groups and tables needed for later analysis
        structure = {
            "acquisition_keys": list(nwb.acquisition.keys()),
            "processing_keys": list(nwb.processing.keys()),
            "interval_keys": list(nwb.intervals.keys()) if nwb.intervals is not None else [],
            "units_count": 0 if nwb.units is None else len(nwb.units.id[:]),
            "unit_columns": [] if nwb.units is None else list(nwb.units.colnames),
            "behavior_interfaces": {},
        }
        # pseudocode: if a behavior module exists, record the child time series under each interface
        if "behavior" in nwb.processing:
            # pseudocode: iterate through every behavior data interface stored in the file
            for name, obj in nwb.processing["behavior"].data_interfaces.items():
                # pseudocode: list named children from time_series interfaces such as velocity
                if hasattr(obj, "time_series"):
                    children = list(obj.time_series.keys())
                # pseudocode: otherwise list named children from spatial_series interfaces such as position
                elif hasattr(obj, "spatial_series"):
                    children = list(obj.spatial_series.keys())
                # pseudocode: fall back to an empty list when the interface has no named children
                else:
                    children = []
                # pseudocode: store the discovered child names under this behavior interface
                structure["behavior_interfaces"][name] = children
        # pseudocode: return the assembled structural summary for reporting and debugging
        return structure


def load_session_arrays(local_path: Path) -> Dict:
    """
    INPUT:
    local_path : path to downloaded NWB file (Path)

    OUTPUT:
    session : dict with vel_timestamps (n_vel,), vel_values (n_vel, 2), pos_values (n_pos, 2), trials (DataFrame), unit_ids (n_units,), spike_times_by_unit (List), subject_id (str | None), session_description (str) (dict)
    """
    # pseudocode: open the NWB file so we can extract arrays for modeling
    with NWBHDF5IO(str(local_path), "r", load_namespaces=True) as io:
        # pseudocode: deserialize the NWB recording into memory
        nwb = io.read()
        # pseudocode: grab the behavior processing module that stores cursor measurements
        behavior = nwb.processing["behavior"]
        # pseudocode: extract the cursor velocity time series
        vel = behavior.data_interfaces["Velocity"].time_series["cursor_vel"]
        # pseudocode: extract the cursor position spatial series
        pos = behavior.data_interfaces["Position"].spatial_series["cursor_pos"]
        # pseudocode: copy the NWB trials table into a mutable pandas DataFrame
        trials = nwb.trials.to_dataframe().copy()
        # pseudocode: keep a reference to the units table for neural metadata
        units = nwb.units
        # pseudocode: convert unit identifiers into strings for stable downstream vocab keys
        unit_ids = np.array([str(x) for x in units.id[:]], dtype=object)
        # pseudocode: materialize one NumPy spike-time array per unit
        spike_times_by_unit = [np.asarray(st, dtype=np.float64) for st in units["spike_times"][:]]
        # pseudocode: return every array and metadata field required by the notebook pipeline
        return {
            "vel_timestamps": np.asarray(vel.timestamps[:], dtype=np.float64),
            "vel_values": np.asarray(vel.data[:], dtype=np.float32),
            "pos_values": np.asarray(pos.data[:], dtype=np.float32),
            "trials": trials,
            "unit_ids": unit_ids,
            "spike_times_by_unit": spike_times_by_unit,
            "subject_id": getattr(getattr(nwb, "subject", None), "subject_id", None),
            "session_description": nwb.session_description,
        }

## 2.7 Trial Selection and Splitting

In [ ]:
def choose_successful_trials(trials: pd.DataFrame, context_sec: float, min_trials: int = 20) -> pd.DataFrame:
    """
    INPUT:
    trials      : raw trial table (DataFrame)
    context_sec : min duration minus margin (float)
    min_trials  : min "R" trials for filtering (int)

    OUTPUT:
    trial_df : filtered trials with "duration_sec" column (DataFrame)
    """
    # pseudocode: work on a copy so the original trial table remains untouched
    trial_df = trials.copy()
    # pseudocode: measure each trial duration from its stop and start timestamps
    duration = trial_df["stop_time"] - trial_df["start_time"]
    # pseudocode: keep only trials that are long enough to hold the full context window
    trial_df = trial_df.loc[duration >= context_sec + 0.05].copy()
    # pseudocode: if enough rewarded trials exist, restrict the dataset to successful trials only
    if "result" in trial_df.columns and (trial_df["result"] == "R").sum() >= min_trials:
        # pseudocode: keep only the rewarded trials
        trial_df = trial_df.loc[trial_df["result"] == "R"].copy()
    # pseudocode: store the final duration explicitly for downstream inspection
    trial_df["duration_sec"] = trial_df["stop_time"] - trial_df["start_time"]
    # pseudocode: return the filtered trial table
    return trial_df


def split_trials(trial_df: pd.DataFrame, seed: int) -> Dict[str, np.ndarray]:
    """
    INPUT:
    trial_df : filtered trial table (DataFrame)
    seed     : random seed (int)

    OUTPUT:
    split_map : dict mapping "train"/"val"/"test" to sorted trial-ID arrays (Dict[str, ndarray])
    """
    # pseudocode: extract the trial identifiers that will be partitioned
    ids = trial_df.index.to_numpy()
    # pseudocode: initialize a reproducible NumPy random generator
    rng = np.random.default_rng(seed)
    # pseudocode: copy the identifiers before shuffling them in place
    shuffled = ids.copy()
    # pseudocode: randomly permute the trial identifiers
    rng.shuffle(shuffled)
    # pseudocode: record how many total trials are available
    n = len(shuffled)
    # pseudocode: allocate roughly twenty percent of trials to the test split
    n_test = max(1, round(0.20 * n))
    # pseudocode: allocate roughly ten percent of trials to the validation split
    n_val = max(1, round(0.10 * n))
    # pseudocode: sort the chosen test identifiers for easier reading downstream
    test_ids = np.sort(shuffled[:n_test])
    # pseudocode: sort the chosen validation identifiers
    val_ids = np.sort(shuffled[n_test : n_test + n_val])
    # pseudocode: use the remaining identifiers for training
    train_ids = np.sort(shuffled[n_test + n_val :])
    # pseudocode: return the split dictionary expected by later preprocessing code
    return {"train": train_ids, "val": val_ids, "test": test_ids}


def build_sample_table(
    vel_timestamps: np.ndarray,
    vel_values: np.ndarray,
    trials: pd.DataFrame,
    split_map: Dict[str, np.ndarray],
    context_sec: float,
    stride_sec: float,
) -> pd.DataFrame:
    """
    INPUT:
    vel_timestamps : velocity sample timestamps (n_vel,)
    vel_values     : velocity vectors aligned to vel_timestamps (n_vel, 2)
    trials         : filtered trial table (DataFrame)
    split_map      : dict mapping split names to trial identifiers (Dict[str, ndarray])
    context_sec    : decoding context window length in seconds (float)
    stride_sec     : temporal spacing between sampled targets in seconds (float)

    OUTPUT:
    sample_df : one row per decodable time-step with columns split, trial_id, time, vel_idx, vx, vy (DataFrame)
    """
    # pseudocode: estimate the median spacing between consecutive velocity timestamps
    dt = float(np.median(np.diff(vel_timestamps)))
    # pseudocode: convert the desired stride from seconds into an integer number of time steps
    stride_steps = max(1, int(round(stride_sec / dt)))
    # pseudocode: accumulate one row per supervised decoding example
    rows = []
    # pseudocode: build a fast lookup set so missing trial IDs can be skipped quickly
    trial_index_lookup = set(trials.index.to_numpy())
    # pseudocode: iterate over each dataset split and its associated trial identifiers
    for split_name, trial_ids in split_map.items():
        # pseudocode: process every trial assigned to the current split
        for trial_id in trial_ids:
            # pseudocode: skip trial IDs that are not present in the filtered trial table
            if trial_id not in trial_index_lookup:
                continue
            # pseudocode: fetch the metadata row for the current trial
            row = trials.loc[trial_id]
            # pseudocode: keep only velocity samples that occur after the context window begins and before the trial ends
            mask = (vel_timestamps >= row["start_time"] + context_sec) & (vel_timestamps <= row["stop_time"])
            # pseudocode: convert the boolean mask into concrete velocity indices
            idx = np.flatnonzero(mask)
            # pseudocode: skip trials that contain no valid target timestamps
            if len(idx) == 0:
                continue
            # pseudocode: subsample the valid indices according to the requested stride
            idx = idx[::stride_steps]
            # pseudocode: turn each retained velocity index into one supervised sample row
            for vel_idx in idx:
                # pseudocode: record the split label, trial ID, timestamp, source index, and 2D velocity target
                rows.append(
                    {
                        "split": split_name,
                        "trial_id": int(trial_id),
                        "time": float(vel_timestamps[vel_idx]),
                        "vel_idx": int(vel_idx),
                        "vx": float(vel_values[vel_idx, 0]),
                        "vy": float(vel_values[vel_idx, 1]),
                    }
                )
    # pseudocode: convert the collected rows into a sorted DataFrame for downstream batching
    sample_df = pd.DataFrame(rows).sort_values(["split", "trial_id", "time"]).reset_index(drop=True)
    # pseudocode: return the final sample table
    return sample_df

## 2.8 Spike Binning and Feature Extraction

In [ ]:
def bin_full_session(
    spike_times_by_unit: List[np.ndarray],
    end_time: float,
    bin_size_sec: float,
) -> np.ndarray:
    """
    INPUT:
    spike_times_by_unit : one spike-time array per unit (List[np.ndarray])
    end_time            : final time to cover when binning the session (float)
    bin_size_sec        : width of each spike-count bin in seconds (float)

    OUTPUT:
    counts : binned spike counts (n_bins, n_units)
    """
    # pseudocode: count how many neural units need their own spike-count column
    n_units = len(spike_times_by_unit)
    # pseudocode: choose enough time bins to cover the entire recording duration
    n_bins = int(math.floor(end_time / bin_size_sec)) + 1
    # pseudocode: initialize the full session spike-count matrix with zeros
    counts = np.zeros((n_bins, n_units), dtype=np.float32)
    # pseudocode: bin spikes for one unit at a time
    for unit_idx, spike_times in enumerate(spike_times_by_unit):
        # pseudocode: convert each spike time into its integer time-bin index
        bin_idx = np.floor(spike_times / bin_size_sec).astype(int)
        # pseudocode: keep only spike bins that fall inside the allocated session range
        valid = (bin_idx >= 0) & (bin_idx < n_bins)
        # pseudocode: add one count for every valid spike in this unit's column
        if valid.any():
            np.add.at(counts[:, unit_idx], bin_idx[valid], 1.0)
    # pseudocode: return the complete spike-count matrix
    return counts


def build_baseline_arrays(
    sample_df: pd.DataFrame,
    binned_counts: np.ndarray,
    context_bins: int,
) -> Tuple[np.ndarray, np.ndarray]:
    """
    INPUT:
    sample_df      : supervised sample table with velocity indices (DataFrame)
    binned_counts  : full-session spike-count matrix (n_bins, n_units)
    context_bins   : number of time bins in each context window (int)

    OUTPUT:
    X : feature tensor (n_samples, context_bins, n_units)
    y : target velocities (n_samples, 2)
    """
    # pseudocode: read the velocity-aligned time-bin indices for every supervised sample
    vel_idx = sample_df["vel_idx"].to_numpy(dtype=int)
    # pseudocode: build the sliding-window bin indices that define each context tensor
    feature_idx = vel_idx[:, None] - context_bins + np.arange(context_bins)
    # pseudocode: gather the spike-count history for every sample window
    X = binned_counts[feature_idx]
    # pseudocode: extract the matching two-dimensional velocity targets
    y = sample_df[["vx", "vy"]].to_numpy(dtype=np.float32)
    # pseudocode: return feature windows and targets as float32 arrays
    return X.astype(np.float32), y

## 2.9 Metrics Helper

In [ ]:
def r2_metrics(y_true: np.ndarray, y_pred: np.ndarray) -> Dict[str, float]:
    """
    INPUT:
    y_true : ground-truth velocities (n_samples, 2)
    y_pred : predicted velocities (n_samples, 2)

    OUTPUT:
    metrics : dict with "r2_mean", "r2_vx", "r2_vy" (Dict[str, float])
    """
    # pseudocode: return one overall R^2 score plus one score for each velocity axis
    return {
        # pseudocode: average performance across the x and y velocity dimensions
        "r2_mean": float(r2_score(y_true, y_pred, multioutput="uniform_average")),
        # pseudocode: score prediction quality along the x-velocity dimension
        "r2_vx": float(r2_score(y_true[:, 0], y_pred[:, 0])),
        # pseudocode: score prediction quality along the y-velocity dimension
        "r2_vy": float(r2_score(y_true[:, 1], y_pred[:, 1])),
    }

## 2.10 Baseline Model Definitions (MLP and GRU)

In [ ]:
class MLPDecoder(nn.Module):
    def __init__(self, input_dim: int, hidden_dim: int = 256):
        """
        INPUT:
        input_dim  : dimensionality of the flattened spike-history vector (int)
        hidden_dim : width of the hidden fully connected layers (int)

        OUTPUT:
        None (constructor)
        """
        # pseudocode: initialize the parent PyTorch module machinery
        super().__init__()
        # pseudocode: build a feedforward network that maps spike-history features to 2D velocity
        self.net = nn.Sequential(
            # pseudocode: project the flattened context window into the hidden feature space
            nn.Linear(input_dim, hidden_dim),
            # pseudocode: introduce nonlinearity after the first projection
            nn.ReLU(),
            # pseudocode: regularize the hidden representation with dropout
            nn.Dropout(0.2),
            # pseudocode: apply a second hidden transformation at the same width
            nn.Linear(hidden_dim, hidden_dim),
            # pseudocode: apply another nonlinearity before the final readout
            nn.ReLU(),
            # pseudocode: regularize the second hidden layer as well
            nn.Dropout(0.2),
            # pseudocode: map the hidden features down to vx and vy outputs
            nn.Linear(hidden_dim, 2),
        )

    def forward(self, x):
        """
        INPUT:
        x : flattened spike-count features (batch_size, input_dim)

        OUTPUT:
        pred : predicted 2D velocity (batch_size, 2)
        """
        # pseudocode: pass the input batch through the multilayer perceptron to predict velocity
        return self.net(x)


class GRUDecoder(nn.Module):
    def __init__(self, input_dim: int, hidden_dim: int = 128):
        """
        INPUT:
        input_dim  : number of spike-count features per time bin (int)
        hidden_dim : dimensionality of the GRU hidden state (int)

        OUTPUT:
        None (constructor)
        """
        # pseudocode: initialize the parent PyTorch module machinery
        super().__init__()
        # pseudocode: build a recurrent layer that summarizes the spike-count sequence over time
        self.gru = nn.GRU(input_dim, hidden_dim, batch_first=True)
        # pseudocode: build a linear readout that converts the final hidden state into vx and vy
        self.readout = nn.Linear(hidden_dim, 2)

    def forward(self, x):
        """
        INPUT:
        x : sequence of binned spike counts (batch_size, context_bins, input_dim)

        OUTPUT:
        pred : predicted 2D velocity from last hidden state (batch_size, 2)
        """
        # pseudocode: run the recurrent network across the context window to obtain hidden states
        out, _ = self.gru(x)
        # pseudocode: read out velocity from the final time step of the GRU output sequence
        return self.readout(out[:, -1])

## 2.11 Baseline Training Utilities

In [ ]:
def iterate_batches(
    loader: DataLoader,
    model: nn.Module,
    optimizer=None,
    target_mean: np.ndarray | None = None,
    target_std: np.ndarray | None = None,
):
    """
    INPUT:
    loader      : mini-batch iterator over features and targets (DataLoader)
    model       : neural decoder to train or evaluate (nn.Module)
    optimizer   : optimizer used for training, or None for evaluation (Optimizer | None)
    target_mean : training-set target mean for de-normalization (1, 2) or None
    target_std  : training-set target standard deviation for de-normalization (1, 2) or None

    OUTPUT:
    metrics    : aggregate loss and R^2 scores for the full loader (Dict[str, float])
    preds_np   : model predictions collected across the loader (n_samples, 2)
    targets_np : target velocities collected across the loader (n_samples, 2)
    """
    # pseudocode: use mean squared error as the regression loss for decoder training
    criterion = nn.MSELoss()
    # pseudocode: switch behavior based on whether an optimizer was supplied
    is_train = optimizer is not None
    # pseudocode: accumulate the total loss across all examples
    total_loss = 0.0
    # pseudocode: collect predictions and targets so we can compute full-dataset metrics
    preds, targets = [], []
    # pseudocode: put the model in training mode when optimizing, otherwise in evaluation mode
    model.train(is_train)
    # pseudocode: iterate through one mini-batch at a time
    for xb, yb in loader:
        # pseudocode: move the input features onto the active compute device
        xb = xb.to(DEVICE)
        # pseudocode: move the target velocities onto the active compute device
        yb = yb.to(DEVICE)
        # pseudocode: enable gradients only during training passes
        with torch.set_grad_enabled(is_train):
            # pseudocode: generate velocity predictions for the current mini-batch
            pred = model(xb)
            # pseudocode: compare predictions against targets with mean squared error
            loss = criterion(pred, yb)
            # pseudocode: if this is a training pass, update the model parameters
            if is_train:
                # pseudocode: clear any stale gradients before backpropagation
                optimizer.zero_grad()
                # pseudocode: backpropagate the current batch loss through the model
                loss.backward()
                # pseudocode: take one optimizer step to improve the parameters
                optimizer.step()
        # pseudocode: accumulate loss in example-weighted form for dataset averaging
        total_loss += loss.item() * xb.size(0)
        # pseudocode: move predictions back to NumPy for metric computation
        pred_np = pred.detach().cpu().numpy()
        # pseudocode: move targets back to NumPy as well
        target_np = yb.detach().cpu().numpy()
        # pseudocode: undo target normalization when de-normalization statistics are available
        if target_mean is not None and target_std is not None:
            # pseudocode: scale and shift predictions back to the original velocity units
            pred_np = pred_np * target_std + target_mean
            # pseudocode: scale and shift targets back to the original velocity units
            target_np = target_np * target_std + target_mean
        # pseudocode: store this batch of predictions for full-dataset metric computation
        preds.append(pred_np)
        # pseudocode: store this batch of targets for full-dataset metric computation
        targets.append(target_np)
    # pseudocode: concatenate all batch predictions into one array
    preds_np = np.concatenate(preds, axis=0)
    # pseudocode: concatenate all batch targets into one array
    targets_np = np.concatenate(targets, axis=0)
    # pseudocode: compute R^2 scores over the entire loader
    metrics = r2_metrics(targets_np, preds_np)
    # pseudocode: add the average per-example loss to the metric dictionary
    metrics["loss"] = total_loss / len(loader.dataset)
    # pseudocode: return aggregate metrics plus the raw predictions and targets
    return metrics, preds_np, targets_np


def train_torch_decoder(
    model: nn.Module,
    train_loader: DataLoader,
    val_loader: DataLoader,
    max_epochs: int,
    lr: float = 1e-3,
    target_mean: np.ndarray | None = None,
    target_std: np.ndarray | None = None,
) -> Tuple[nn.Module, List[Dict]]:
    """
    INPUT:
    model       : neural decoder to optimize (nn.Module)
    train_loader: mini-batches for training (DataLoader)
    val_loader  : mini-batches for validation (DataLoader)
    max_epochs  : number of training epochs to run (int)
    lr          : optimizer learning rate (float)
    target_mean : training-set target mean for de-normalization (1, 2) or None
    target_std  : training-set target standard deviation for de-normalization (1, 2) or None

    OUTPUT:
    model   : trained model restored to the best validation checkpoint (nn.Module)
    history : per-epoch training summary rows (List[Dict])
    """
    # pseudocode: initialize AdamW with mild weight decay for stable decoder training
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    # pseudocode: keep a row of metrics for every epoch
    history = []
    # pseudocode: store the best-performing validation checkpoint seen so far
    best_state = None
    # pseudocode: initialize the best validation R^2 score to negative infinity
    best_val = -np.inf
    # pseudocode: move the model parameters onto the active compute device
    model.to(DEVICE)
    # pseudocode: train for the requested number of epochs
    for epoch in range(1, max_epochs + 1):
        # pseudocode: run one full training epoch and collect de-normalized metrics
        train_metrics, _, _ = iterate_batches(
            train_loader,
            model,
            optimizer,
            target_mean=target_mean,
            target_std=target_std,
        )
        # pseudocode: evaluate the model on the validation split without updating weights
        val_metrics, _, _ = iterate_batches(
            val_loader,
            model,
            optimizer=None,
            target_mean=target_mean,
            target_std=target_std,
        )
        # pseudocode: summarize the key losses and R^2 scores for this epoch
        row = {
            "epoch": epoch,
            "train_loss": train_metrics["loss"],
            "val_loss": val_metrics["loss"],
            "train_r2_mean": train_metrics["r2_mean"],
            "val_r2_mean": val_metrics["r2_mean"],
        }
        # pseudocode: append the epoch summary to the training history
        history.append(row)
        # pseudocode: whenever validation improves, snapshot the current model weights
        if val_metrics["r2_mean"] > best_val:
            # pseudocode: record the new best validation R^2 score
            best_val = val_metrics["r2_mean"]
            # pseudocode: deep-copy the model state so it can be restored later
            best_state = copy.deepcopy(model.state_dict())
    # pseudocode: restore the best validation checkpoint after training finishes
    if best_state is not None:
        # pseudocode: load the saved best-performing weights back into the model
        model.load_state_dict(best_state)
    # pseudocode: return the best checkpoint together with the epoch-by-epoch history
    return model, history

## 2.12 POYO Single-Query Dataset

In [ ]:
class POYOSingleQueryDataset(Dataset):
    def __init__(
        self,
        sample_df: pd.DataFrame,
        spike_times_by_unit: List[np.ndarray],
        unit_ids: np.ndarray,
        session_id: str,
        model: POYO,
        context_sec: float = CONTEXT_SEC,
        normalize_std: float = 20.0,
    ):
        """
        INPUT:
        sample_df           : one row per supervised decoding target (DataFrame)
        spike_times_by_unit : one spike-time array per unit (List[np.ndarray])
        unit_ids            : string identifiers for each recorded unit (n_units,)
        session_id          : identifier used for the POYO session embedding (str)
        model               : tokenizing POYO model instance (POYO)
        context_sec         : amount of spike history to include before each target (float)
        normalize_std       : standard deviation used for target normalization (float)

        OUTPUT:
        None (constructor)
        """
        # pseudocode: store a reset-index copy of the sample table for stable integer indexing
        self.sample_df = sample_df.reset_index(drop=True)
        # pseudocode: keep the spike trains so each sample can pull its local spike window
        self.spike_times_by_unit = spike_times_by_unit
        # pseudocode: keep the unit identifiers for POYO's unit embedding vocabulary
        self.unit_ids = unit_ids
        # pseudocode: store the session identifier for POYO's session embedding vocabulary
        self.session_id = session_id
        # pseudocode: keep the POYO model so we can call its tokenizer inside __getitem__
        self.model = model
        # pseudocode: store the history window length in seconds
        self.context_sec = context_sec
        # pseudocode: store the output normalization scale expected by the readout config
        self.normalize_std = normalize_std

    def __len__(self):
        """
        INPUT:
        None

        OUTPUT:
        length : number of samples (int)
        """
        # pseudocode: report how many supervised samples are available in the dataset
        return len(self.sample_df)

    def __getitem__(self, idx):
        """
        INPUT:
        idx : sample index (int)

        OUTPUT:
        tokenized : dict of tensors for POYO forward pass (dict)
        """
        # pseudocode: fetch the metadata row for the requested decoding sample
        row = self.sample_df.iloc[idx]
        # pseudocode: treat the sample timestamp as the right edge of the context window
        end = float(row["time"])
        # pseudocode: compute the left edge of the context window
        start = end - self.context_sec
        # pseudocode: collect spike timestamps from all units that fired within the window
        spike_timestamps = []
        # pseudocode: collect the matching unit indices for those spike timestamps
        spike_unit_index = []
        # pseudocode: scan each unit's spike train to extract spikes that fall inside the context window
        for unit_idx, spike_times in enumerate(self.spike_times_by_unit):
            # pseudocode: find the first spike at or after the window start with binary search
            left = np.searchsorted(spike_times, start, side="left")
            # pseudocode: find the first spike strictly after the window end with binary search
            right = np.searchsorted(spike_times, end, side="right")
            # pseudocode: if this unit fired in the window, store its relative spike times and unit labels
            if right > left:
                # pseudocode: shift spikes so the window start becomes time zero
                rel = spike_times[left:right] - start
                # pseudocode: append the relative spike times for this unit
                spike_timestamps.append(rel.astype(np.float64))
                # pseudocode: append a matching array of the current unit index
                spike_unit_index.append(np.full(len(rel), unit_idx, dtype=np.int64))
        # pseudocode: if any spikes were found, concatenate and time-sort them across units
        if spike_timestamps:
            # pseudocode: merge all unit-specific spike times into one timestamp array
            spike_timestamps_arr = np.concatenate(spike_timestamps)
            # pseudocode: merge all unit indices into one parallel array
            spike_unit_index_arr = np.concatenate(spike_unit_index)
            # pseudocode: sort spikes by time while preserving the order of ties
            order = np.argsort(spike_timestamps_arr, kind="stable")
            # pseudocode: reorder the timestamps into chronological order
            spike_timestamps_arr = spike_timestamps_arr[order]
            # pseudocode: reorder the matching unit indices using the same permutation
            spike_unit_index_arr = spike_unit_index_arr[order]
        # pseudocode: otherwise create empty arrays to represent a silent context window
        else:
            # pseudocode: create an empty timestamp array for the no-spike case
            spike_timestamps_arr = np.empty((0,), dtype=np.float64)
            # pseudocode: create an empty unit-index array for the no-spike case
            spike_unit_index_arr = np.empty((0,), dtype=np.int64)

        # pseudocode: bundle spikes, targets, metadata, and configuration into the structure expected by POYO
        data = MiniData(
            spikes=IrregularTimeSeries(
                timestamps=spike_timestamps_arr,
                unit_index=spike_unit_index_arr,
                domain="auto",
            ),
            cursor=IrregularTimeSeries(
                timestamps=np.array([self.context_sec], dtype=np.float64),
                vel=np.array([[row["vx"], row["vy"]]], dtype=np.float32),
                domain="auto",
            ),
            units=ArrayDict(id=self.unit_ids.copy()),
            session=SimpleSessionDescription(id=self.session_id),
            config={
                "readout": {
                    "readout_id": "cursor_velocity_2d",
                    "normalize_mean": 0.0,
                    "normalize_std": self.normalize_std,
                }
            },
            absolute_start=float(start),
            domain=Interval(0.0, self.context_sec),
        )
        # pseudocode: tokenize the structured sample into tensors ready for POYO's forward pass
        return self.model.tokenize(data)

## 2.13 POYO Training and Evaluation

In [ ]:
def evaluate_poyo_loader(model: POYO, loader: DataLoader, denorm_std: float = 20.0):
    """
    INPUT:
    model      : trained POYO model to evaluate (POYO)
    loader     : mini-batches of tokenized POYO samples (DataLoader)
    denorm_std : standard deviation used to undo POYO target normalization (float)

    OUTPUT:
    metrics    : aggregate loss and R^2 scores for the loader (Dict[str, float])
    preds_np   : denormalized POYO predictions (n_samples, 2)
    targets_np : denormalized target velocities (n_samples, 2)
    """
    # pseudocode: put POYO in evaluation mode so dropout and similar layers behave deterministically
    model.eval()
    # pseudocode: accumulate denormalized predictions and targets across all batches
    preds, targets = [], []
    # pseudocode: accumulate example-weighted loss across the full loader
    total_loss = 0.0
    # pseudocode: use elementwise MSE so target weights can be applied after masking
    criterion = nn.MSELoss(reduction="none")
    # pseudocode: disable gradient tracking during evaluation
    with torch.no_grad():
        # pseudocode: process one tokenized batch at a time
        for batch in loader:
            # pseudocode: move every tensor inside the model input dictionary onto the active device
            model_inputs = {k: v.to(DEVICE) if torch.is_tensor(v) else v for k, v in batch["model_inputs"].items()}
            # pseudocode: move the target values to the active device
            target_values = batch["target_values"].to(DEVICE)
            # pseudocode: move the per-target weights to the active device
            target_weights = batch["target_weights"].to(DEVICE)
            # pseudocode: read the boolean mask that identifies valid output positions
            output_mask = batch["model_inputs"]["output_mask"].to(DEVICE)
            # pseudocode: run the POYO forward pass on the tokenized inputs
            output = model(**model_inputs)
            # pseudocode: keep only the output rows that correspond to real prediction targets
            masked_output = output[output_mask]
            # pseudocode: keep the matching target rows
            masked_target = target_values[output_mask]
            # pseudocode: keep the matching per-target weights
            masked_weights = target_weights[output_mask]
            # pseudocode: compute elementwise squared error on the valid target positions
            loss = criterion(masked_output, masked_target)
            # pseudocode: weight and average the masked losses into one scalar batch loss
            loss = (loss * masked_weights.unsqueeze(-1)).mean()
            # pseudocode: accumulate example-weighted loss for the entire dataset
            total_loss += loss.item() * target_values.shape[0]
            # pseudocode: de-normalize and store the valid predictions for R^2 computation
            preds.append((masked_output.cpu().numpy() * denorm_std).reshape(-1, 2))
            # pseudocode: de-normalize and store the valid targets for R^2 computation
            targets.append((masked_target.cpu().numpy() * denorm_std).reshape(-1, 2))
    # pseudocode: concatenate predictions from every batch into one array
    preds_np = np.concatenate(preds, axis=0)
    # pseudocode: concatenate targets from every batch into one array
    targets_np = np.concatenate(targets, axis=0)
    # pseudocode: compute R^2 scores over the full evaluation split
    metrics = r2_metrics(targets_np, preds_np)
    # pseudocode: add the average per-example loss to the metrics
    metrics["loss"] = total_loss / len(loader.dataset)
    # pseudocode: return aggregate metrics plus the denormalized predictions and targets
    return metrics, preds_np, targets_np


def train_poyo_model(
    model: POYO,
    train_loader: DataLoader,
    val_loader: DataLoader,
    max_epochs: int,
) -> Tuple[POYO, List[Dict]]:
    """
    INPUT:
    model       : POYO model to optimize (POYO)
    train_loader: mini-batches for training (DataLoader)
    val_loader  : mini-batches for validation (DataLoader)
    max_epochs  : number of training epochs to run (int)

    OUTPUT:
    model   : trained POYO model restored to the best validation checkpoint (POYO)
    history : per-epoch training summary rows (List[Dict])
    """
    # pseudocode: move the POYO parameters onto the active compute device
    model.to(DEVICE)
    # pseudocode: collect sparse embedding parameters that need special optimizer handling
    special_emb_params = list(model.unit_emb.parameters()) + list(model.session_emb.parameters())
    # pseudocode: collect every remaining dense parameter in the model
    remaining_params = [
        p for n, p in model.named_parameters() if "unit_emb" not in n and "session_emb" not in n
    ]
    # pseudocode: configure SparseLamb with separate sparse and dense parameter groups
    optimizer = SparseLamb(
        [
            {"params": special_emb_params, "sparse": True},
            {"params": remaining_params},
        ],
        lr=POYO_LR,
        weight_decay=POYO_WEIGHT_DECAY,
    )
    # pseudocode: record one summary row per epoch
    history = []
    # pseudocode: remember the best validation checkpoint seen so far
    best_state = None
    # pseudocode: initialize the best validation R^2 score to negative infinity
    best_val = -np.inf
    # pseudocode: use elementwise MSE so target weights can be applied after masking
    criterion = nn.MSELoss(reduction="none")

    # pseudocode: loop over the requested number of training epochs
    for epoch in range(1, max_epochs + 1):
        # pseudocode: put the model into training mode before the optimization pass
        model.train()
        # pseudocode: accumulate example-weighted training loss across the epoch
        train_loss_total = 0.0
        # pseudocode: collect denormalized training predictions and targets for metrics
        train_preds, train_targets = [], []
        # pseudocode: iterate through each training batch
        for batch in train_loader:
            # pseudocode: move every tensor inside the model input dictionary onto the active device
            model_inputs = {k: v.to(DEVICE) if torch.is_tensor(v) else v for k, v in batch["model_inputs"].items()}
            # pseudocode: move the target values to the active device
            target_values = batch["target_values"].to(DEVICE)
            # pseudocode: move the per-target weights to the active device
            target_weights = batch["target_weights"].to(DEVICE)
            # pseudocode: read the boolean mask that identifies valid output positions
            output_mask = batch["model_inputs"]["output_mask"].to(DEVICE)
            # pseudocode: run the POYO forward pass on the tokenized inputs
            output = model(**model_inputs)
            # pseudocode: keep only the output rows that correspond to real prediction targets
            masked_output = output[output_mask]
            # pseudocode: keep the matching target rows
            masked_target = target_values[output_mask]
            # pseudocode: keep the matching per-target weights
            masked_weights = target_weights[output_mask]
            # pseudocode: compute elementwise squared error on the valid target positions
            loss = criterion(masked_output, masked_target)
            # pseudocode: weight and average the masked losses into one scalar batch loss
            loss = (loss * masked_weights.unsqueeze(-1)).mean()
            # pseudocode: clear old gradients before backpropagation
            optimizer.zero_grad()
            # pseudocode: backpropagate the weighted loss through POYO
            loss.backward()
            # pseudocode: update the sparse and dense parameters with one optimizer step
            optimizer.step()
            # pseudocode: accumulate example-weighted loss for the full training split
            train_loss_total += loss.item() * target_values.shape[0]
            # pseudocode: de-normalize and store training predictions for R^2 computation
            train_preds.append((masked_output.detach().cpu().numpy() * 20.0).reshape(-1, 2))
            # pseudocode: de-normalize and store training targets for R^2 computation
            train_targets.append((masked_target.detach().cpu().numpy() * 20.0).reshape(-1, 2))
        # pseudocode: concatenate all training predictions into one array
        train_preds_np = np.concatenate(train_preds, axis=0)
        # pseudocode: concatenate all training targets into one array
        train_targets_np = np.concatenate(train_targets, axis=0)
        # pseudocode: compute training R^2 scores for the epoch
        train_metrics = r2_metrics(train_targets_np, train_preds_np)
        # pseudocode: add the average training loss to the training metrics
        train_metrics["loss"] = train_loss_total / len(train_loader.dataset)

        # pseudocode: evaluate the current POYO checkpoint on the validation split
        val_metrics, _, _ = evaluate_poyo_loader(model, val_loader, denorm_std=20.0)
        # pseudocode: append a concise summary of the epoch's train and validation metrics
        history.append(
            {
                "epoch": epoch,
                "train_loss": train_metrics["loss"],
                "val_loss": val_metrics["loss"],
                "train_r2_mean": train_metrics["r2_mean"],
                "val_r2_mean": val_metrics["r2_mean"],
            }
        )
        # pseudocode: snapshot the model whenever validation R^2 improves
        if val_metrics["r2_mean"] > best_val:
            # pseudocode: remember the new best validation score
            best_val = val_metrics["r2_mean"]
            # pseudocode: deep-copy the current parameter state for later restoration
            best_state = copy.deepcopy(model.state_dict())
    # pseudocode: restore the best validation checkpoint after training completes
    if best_state is not None:
        # pseudocode: load the saved best-performing parameter state
        model.load_state_dict(best_state)
    # pseudocode: return the best checkpoint together with the training history
    return model, history


def make_tensor_loader(X: np.ndarray, y: np.ndarray, batch_size: int, shuffle: bool) -> DataLoader:
    """
    INPUT:
    X          : feature array to batch (n_samples, ...)
    y          : target velocity array (n_samples, 2)
    batch_size : number of examples per batch (int)
    shuffle    : whether to randomize the batch order (bool)

    OUTPUT:
    loader : mini-batch iterator over X and y (DataLoader)
    """
    # pseudocode: wrap the feature array as a PyTorch tensor without copying data unnecessarily
    tensor_x = torch.from_numpy(X)
    # pseudocode: wrap the target array as a PyTorch tensor
    tensor_y = torch.from_numpy(y)
    # pseudocode: pair features and targets into a tensor dataset
    ds = TensorDataset(tensor_x, tensor_y)
    # pseudocode: return a DataLoader that serves mini-batches from the tensor dataset
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle)

## 2.14 Plotting Utilities

In [ ]:
def plot_learning_curve(history: List[Dict], title: str, out_path: Path) -> None:
    """
    INPUT:
    history  : per-epoch training summaries (List[Dict])
    title    : figure title to display above the plot (str)
    out_path : file path where the figure should be saved (Path)

    OUTPUT:
    None (saves figure)
    """
    # pseudocode: convert the epoch history into a DataFrame for convenient plotting
    hist = pd.DataFrame(history)
    # pseudocode: create the figure and primary axis for the loss curves
    fig, ax = plt.subplots(figsize=(6, 4))
    # pseudocode: plot training loss versus epoch on the left axis
    ax.plot(hist["epoch"], hist["train_loss"], label="train loss")
    # pseudocode: plot validation loss versus epoch on the left axis
    ax.plot(hist["epoch"], hist["val_loss"], label="val loss")
    # pseudocode: create a secondary axis to show validation R^2 on a separate scale
    ax2 = ax.twinx()
    # pseudocode: plot validation R^2 versus epoch on the right axis
    ax2.plot(hist["epoch"], hist["val_r2_mean"], color="green", label="val R^2")
    # pseudocode: label the chart and both axes for readability
    ax.set_title(title)
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Loss")
    ax2.set_ylabel("Val R^2")
    # pseudocode: gather legend handles from the left-axis artists
    lines, labels = ax.get_legend_handles_labels()
    # pseudocode: gather legend handles from the right-axis artists
    lines2, labels2 = ax2.get_legend_handles_labels()
    # pseudocode: merge both legends into a single combined legend
    ax.legend(lines + lines2, labels + labels2, loc="center right")
    # pseudocode: tighten the layout so labels do not overlap
    fig.tight_layout()
    # pseudocode: save the learning-curve figure to disk
    fig.savefig(out_path, dpi=160)
    # pseudocode: close the figure to release notebook memory
    plt.close(fig)


def plot_pred_vs_actual(sample_df: pd.DataFrame, y_pred: np.ndarray, model_name: str, out_path: Path) -> None:
    """
    INPUT:
    sample_df   : held-out sample table containing times and true velocities (DataFrame)
    y_pred      : predicted velocities aligned to sample_df (n_samples, 2)
    model_name  : display name of the model being visualized (str)
    out_path    : file path where the figure should be saved (Path)

    OUTPUT:
    None (saves figure)
    """
    # pseudocode: choose the first held-out trial to visualize as a continuous time trace
    plot_trial = int(sample_df["trial_id"].iloc[0])
    # pseudocode: isolate all held-out samples from that chosen trial
    plot_df = sample_df.loc[sample_df["trial_id"] == plot_trial].copy().reset_index(drop=True)
    # pseudocode: limit the plot to at most 150 aligned time steps
    n = min(len(plot_df), len(y_pred), 150)
    # pseudocode: extract the timestamps for the chosen plotting window
    t = plot_df["time"].to_numpy()[:n]
    # pseudocode: extract the ground-truth velocity components
    y_true = plot_df[["vx", "vy"]].to_numpy()[:n]
    # pseudocode: slice the predicted velocity components to the same plotting window
    y_hat = y_pred[:n]
    # pseudocode: create one subplot for vx and one for vy
    fig, axes = plt.subplots(2, 1, figsize=(9, 5), sharex=True)
    # pseudocode: plot actual and predicted vx over time
    axes[0].plot(t, y_true[:, 0], label="actual vx")
    axes[0].plot(t, y_hat[:, 0], label="pred vx", alpha=0.8)
    axes[0].legend(loc="upper right")
    axes[0].set_ylabel("vx (cm/s)")
    # pseudocode: plot actual and predicted vy over time
    axes[1].plot(t, y_true[:, 1], label="actual vy")
    axes[1].plot(t, y_hat[:, 1], label="pred vy", alpha=0.8)
    axes[1].legend(loc="upper right")
    axes[1].set_ylabel("vy (cm/s)")
    axes[1].set_xlabel("Time (s)")
    # pseudocode: title the figure with the current model name
    fig.suptitle(f"{model_name}: predicted vs actual on one held-out trial")
    # pseudocode: tighten the layout, save the plot, and release the figure
    fig.tight_layout()
    fig.savefig(out_path, dpi=160)
    plt.close(fig)

## 2.15 Standalone Entry Point (not used by notebook)

In [ ]:
def main():
    """
    INPUT:
    None : reads global config constants from the notebook state (none -- reads global config constants)

    OUTPUT:
    None (writes artifacts to ARTIFACTS_DIR)
    """
    # pseudocode: --- Phase: Reproducibility and Data Loading ---
    # pseudocode: seed every random number generator so splits and training runs are repeatable
    set_seed(RANDOM_SEED)

    # pseudocode: choose the smallest DANDI asset that still passes the notebook quality checks
    selected = download_smallest_valid_asset()
    # pseudocode: inspect the NWB structure for later reporting and debugging
    structure = inspect_nwb_file(selected.local_path)
    # pseudocode: load the neural, behavioral, and trial arrays used by the rest of the pipeline
    session = load_session_arrays(selected.local_path)

    # pseudocode: --- Phase: Trial Filtering and Sample Construction ---
    # pseudocode: keep only trials that are long enough and, when possible, successful
    trial_df = choose_successful_trials(session["trials"], context_sec=CONTEXT_SEC)
    # pseudocode: split trials into train, validation, and test partitions
    split_map = split_trials(trial_df, RANDOM_SEED)
    # pseudocode: turn each decodable timestamp into one supervised training row
    sample_df = build_sample_table(
        session["vel_timestamps"],
        session["vel_values"],
        trial_df,
        split_map,
        context_sec=CONTEXT_SEC,
        stride_sec=TARGET_STRIDE_SEC,
    )

    # pseudocode: --- Phase: Spike Binning and Baseline Feature Preparation ---
    # pseudocode: find the latest time reached by either the velocity trace or any spike train
    end_time = max(session["vel_timestamps"][-1], max(st[-1] for st in session["spike_times_by_unit"] if len(st)))
    # pseudocode: bin the full recording session into fixed-width spike-count bins
    binned_counts = bin_full_session(session["spike_times_by_unit"], end_time=end_time, bin_size_sec=BIN_SIZE_SEC)
    # pseudocode: convert the context window length from seconds into a number of bins
    context_bins = int(round(CONTEXT_SEC / BIN_SIZE_SEC))

    # pseudocode: gather one DataFrame per split so later code can address train, val, and test separately
    split_frames = {
        split: sample_df.loc[sample_df["split"] == split].copy().reset_index(drop=True)
        for split in ["train", "val", "test"]
    }
    # pseudocode: optionally trim each split for fast development-mode experiments
    if FAST_DEV_RUN:
        # pseudocode: cap the number of samples per split according to the fast-dev configuration
        for split, limit in FAST_DEV_MAX_SAMPLES.items():
            split_frames[split] = split_frames[split].head(limit).copy().reset_index(drop=True)

    # pseudocode: materialize baseline-model features and targets for each split
    X_train, y_train = build_baseline_arrays(split_frames["train"], binned_counts, context_bins)
    X_val, y_val = build_baseline_arrays(split_frames["val"], binned_counts, context_bins)
    X_test, y_test = build_baseline_arrays(split_frames["test"], binned_counts, context_bins)

    # pseudocode: flatten the temporal context windows for linear and MLP baselines
    X_train_flat = X_train.reshape(len(X_train), -1)
    X_val_flat = X_val.reshape(len(X_val), -1)
    X_test_flat = X_test.reshape(len(X_test), -1)
    # pseudocode: compute feature normalization statistics from the training split only
    feat_mean = X_train_flat.mean(axis=0, keepdims=True)
    feat_std = X_train_flat.std(axis=0, keepdims=True) + 1e-6
    # pseudocode: normalize flattened features using the training-set mean and standard deviation
    X_train_flat_n = (X_train_flat - feat_mean) / feat_std
    X_val_flat_n = (X_val_flat - feat_mean) / feat_std
    X_test_flat_n = (X_test_flat - feat_mean) / feat_std
    # pseudocode: reshape normalized features back into sequences for the GRU baseline
    X_train_seq_n = X_train_flat_n.reshape(len(X_train), context_bins, X_train.shape[-1])
    X_val_seq_n = X_val_flat_n.reshape(len(X_val), context_bins, X_val.shape[-1])
    X_test_seq_n = X_test_flat_n.reshape(len(X_test), context_bins, X_test.shape[-1])
    # pseudocode: compute target normalization statistics from the training split only
    y_mean = y_train.mean(axis=0, keepdims=True)
    y_std = y_train.std(axis=0, keepdims=True) + 1e-6
    # pseudocode: normalize targets for the neural-network baselines
    y_train_n = ((y_train - y_mean) / y_std).astype(np.float32)
    y_val_n = ((y_val - y_mean) / y_std).astype(np.float32)
    y_test_n = ((y_test - y_mean) / y_std).astype(np.float32)

    # pseudocode: initialize the results table that will hold test-set R^2 scores for every model
    results = {}

    # pseudocode: --- Phase: Wiener Filter Baseline ---
    # pseudocode: instantiate a ridge-regression decoder as the classic Wiener-filter baseline
    ridge = Ridge(alpha=1.0)
    # pseudocode: fit the linear decoder on normalized flattened spike-history features
    ridge.fit(X_train_flat_n, y_train)
    # pseudocode: predict test-set velocities with the fitted linear model
    ridge_pred = ridge.predict(X_test_flat_n).astype(np.float32)
    # pseudocode: store the Wiener baseline test metrics
    results["Wiener"] = r2_metrics(y_test, ridge_pred)
    # pseudocode: visualize predicted versus actual velocities on one held-out trial
    plot_pred_vs_actual(split_frames["test"], ridge_pred, "Wiener", ARTIFACTS_DIR / "wiener_pred_vs_actual.png")

    # pseudocode: --- Phase: MLP Baseline ---
    # pseudocode: instantiate the multilayer perceptron decoder
    mlp = MLPDecoder(input_dim=X_train_flat_n.shape[1]).to(DEVICE)
    # pseudocode: build DataLoaders for MLP training, validation, and testing
    mlp_train_loader = make_tensor_loader(X_train_flat_n.astype(np.float32), y_train_n, MLP_BATCH_SIZE, True)
    mlp_val_loader = make_tensor_loader(X_val_flat_n.astype(np.float32), y_val_n, MLP_BATCH_SIZE, False)
    mlp_test_loader = make_tensor_loader(X_test_flat_n.astype(np.float32), y_test_n, MLP_BATCH_SIZE, False)
    # pseudocode: train the MLP and keep the best validation checkpoint
    mlp, mlp_hist = train_torch_decoder(
        mlp,
        mlp_train_loader,
        mlp_val_loader,
        MAX_EPOCHS_MLP,
        lr=3e-4,
        target_mean=y_mean,
        target_std=y_std,
    )
    # pseudocode: evaluate the trained MLP on the test split in original velocity units
    mlp_test_metrics, mlp_pred, _ = iterate_batches(
        mlp_test_loader,
        mlp,
        optimizer=None,
        target_mean=y_mean,
        target_std=y_std,
    )
    # pseudocode: keep only the MLP R^2 metrics in the comparison table
    results["MLP"] = {k: float(v) for k, v in mlp_test_metrics.items() if k.startswith("r2")}
    # pseudocode: save the MLP learning curve figure
    plot_learning_curve(mlp_hist, "MLP learning curve", ARTIFACTS_DIR / "mlp_learning_curve.png")
    # pseudocode: save an example MLP prediction-versus-actual trace
    plot_pred_vs_actual(split_frames["test"], mlp_pred, "MLP", ARTIFACTS_DIR / "mlp_pred_vs_actual.png")

    # pseudocode: --- Phase: GRU Baseline ---
    # pseudocode: instantiate the recurrent GRU decoder
    gru = GRUDecoder(input_dim=X_train_seq_n.shape[-1]).to(DEVICE)
    # pseudocode: build DataLoaders for GRU training, validation, and testing
    gru_train_loader = make_tensor_loader(X_train_seq_n.astype(np.float32), y_train_n, GRU_BATCH_SIZE, True)
    gru_val_loader = make_tensor_loader(X_val_seq_n.astype(np.float32), y_val_n, GRU_BATCH_SIZE, False)
    gru_test_loader = make_tensor_loader(X_test_seq_n.astype(np.float32), y_test_n, GRU_BATCH_SIZE, False)
    # pseudocode: train the GRU and keep the best validation checkpoint
    gru, gru_hist = train_torch_decoder(
        gru,
        gru_train_loader,
        gru_val_loader,
        MAX_EPOCHS_GRU,
        lr=3e-4,
        target_mean=y_mean,
        target_std=y_std,
    )
    # pseudocode: evaluate the trained GRU on the test split in original velocity units
    gru_test_metrics, gru_pred, _ = iterate_batches(
        gru_test_loader,
        gru,
        optimizer=None,
        target_mean=y_mean,
        target_std=y_std,
    )
    # pseudocode: keep only the GRU R^2 metrics in the comparison table
    results["GRU"] = {k: float(v) for k, v in gru_test_metrics.items() if k.startswith("r2")}
    # pseudocode: save the GRU learning curve figure
    plot_learning_curve(gru_hist, "GRU learning curve", ARTIFACTS_DIR / "gru_learning_curve.png")
    # pseudocode: save an example GRU prediction-versus-actual trace
    plot_pred_vs_actual(split_frames["test"], gru_pred, "GRU", ARTIFACTS_DIR / "gru_pred_vs_actual.png")

    # pseudocode: --- Phase: POYO Baseline ---
    # pseudocode: look up the readout specification for two-dimensional cursor velocity
    readout_spec = MODALITY_REGISTRY["cursor_velocity_2d"]
    # pseudocode: instantiate the POYO architecture used in the project
    poyo = POYO(
        sequence_length=1.0,
        latent_step=0.125,
        num_latents_per_step=16,
        dim=64,
        depth=6,
        dim_head=64,
        cross_heads=2,
        self_heads=8,
        ffn_dropout=0.2,
        lin_dropout=0.4,
        atn_dropout=0.2,
        readout_spec=readout_spec,
    )
    # pseudocode: derive a clean session identifier from the selected asset filename
    session_id = Path(selected.path).stem.replace("_behavior+ecephys", "")
    # pseudocode: register every unit ID with POYO's unit embedding vocabulary
    poyo.unit_emb.initialize_vocab(session["unit_ids"])
    # pseudocode: register the session ID with POYO's session embedding vocabulary
    poyo.session_emb.initialize_vocab([session_id])
    # pseudocode: build POYO datasets that tokenize one query per supervised sample
    train_ds = POYOSingleQueryDataset(split_frames["train"], session["spike_times_by_unit"], session["unit_ids"], session_id, poyo)
    val_ds = POYOSingleQueryDataset(split_frames["val"], session["spike_times_by_unit"], session["unit_ids"], session_id, poyo)
    test_ds = POYOSingleQueryDataset(split_frames["test"], session["spike_times_by_unit"], session["unit_ids"], session_id, poyo)
    # pseudocode: wrap the POYO datasets in DataLoaders that use torch-brain's custom collate function
    train_loader = DataLoader(train_ds, batch_size=POYO_BATCH_SIZE, shuffle=True, collate_fn=tb_collate)
    val_loader = DataLoader(val_ds, batch_size=POYO_BATCH_SIZE, shuffle=False, collate_fn=tb_collate)
    test_loader = DataLoader(test_ds, batch_size=POYO_BATCH_SIZE, shuffle=False, collate_fn=tb_collate)
    # pseudocode: train POYO and keep the best validation checkpoint
    poyo, poyo_hist = train_poyo_model(poyo, train_loader, val_loader, MAX_EPOCHS_POYO)
    # pseudocode: evaluate the trained POYO model on the held-out test split
    poyo_test_metrics, poyo_pred, _ = evaluate_poyo_loader(poyo, test_loader, denorm_std=20.0)
    # pseudocode: keep only the POYO R^2 metrics in the comparison table
    results["POYO"] = {k: float(v) for k, v in poyo_test_metrics.items() if k.startswith("r2")}
    # pseudocode: save the POYO learning curve figure
    plot_learning_curve(poyo_hist, "POYO learning curve", ARTIFACTS_DIR / "poyo_learning_curve.png")
    # pseudocode: save an example POYO prediction-versus-actual trace
    plot_pred_vs_actual(split_frames["test"], poyo_pred, "POYO", ARTIFACTS_DIR / "poyo_pred_vs_actual.png")

    # pseudocode: --- Phase: Serialize Results and Metadata ---
    # pseudocode: convert the model comparison dictionary into a sorted table for easy viewing
    comparison_df = pd.DataFrame(results).T.sort_index()
    # pseudocode: write the comparison table to CSV inside the artifacts directory
    comparison_df.to_csv(ARTIFACTS_DIR / "comparison_metrics.csv")

    # pseudocode: write a JSON summary with selected-asset metadata, NWB structure, and model metrics
    with open(ARTIFACTS_DIR / "metrics.json", "w") as f:
        json.dump(
            {
                "selected_asset": selected.__dict__,
                "nwb_structure": structure,
                "results": results,
                "sample_counts": {k: int(len(v)) for k, v in split_frames.items()},
                "fast_dev_run": FAST_DEV_RUN,
            },
            f,
            indent=2,
            default=json_default,
        )

    # pseudocode: write a second JSON file that captures run configuration and provenance
    with open(ARTIFACTS_DIR / "run_metadata.json", "w") as f:
        json.dump(
            {
                "device": DEVICE,
                "random_seed": RANDOM_SEED,
                "context_sec": CONTEXT_SEC,
                "bin_size_sec": BIN_SIZE_SEC,
                "target_stride_sec": TARGET_STRIDE_SEC,
                "max_epochs": {
                    "mlp": MAX_EPOCHS_MLP,
                    "gru": MAX_EPOCHS_GRU,
                    "poyo": MAX_EPOCHS_POYO,
                },
                "selected_asset": selected.__dict__,
            },
            f,
            indent=2,
            default=json_default,
        )

    # pseudocode: print a concise summary of the selected asset and final comparison table
    print("Selected asset:", selected)
    print(comparison_df)
    print(f"Artifacts written to: {ARTIFACTS_DIR}")

# Part 3: DANDI Data Download / Stream + NWB Inspection

In [ ]:
set_seed(RANDOM_SEED)
selected = download_smallest_valid_asset(candidate_limit=MAX_CANDIDATES_TO_CHECK)
structure = inspect_nwb_file(selected.local_path)

provenance_df = pd.DataFrame(
    [
        {
            "dandiset_id": selected.dandiset_id,
            "published_version": selected.version_id,
            "asset_path": selected.path,
            "asset_size_mb": round(selected.size_bytes / 1e6, 3),
            "subject_id": selected.subject_id,
            "session_description": selected.session_description,
            "duration_sec": round(selected.duration_sec, 2),
            "units_count": selected.units_count,
            "trials_count": selected.trials_count,
            "local_path": str(selected.local_path),
        }
    ]
)
display(provenance_df)
print("Verified NWB structure:")
print(json.dumps(structure, indent=2, default=json_default))


In [ ]:
from pynwb import NWBHDF5IO

with NWBHDF5IO(str(selected.local_path), "r", load_namespaces=True) as io:
    nwb = io.read()
    print("Behavior interfaces:", list(nwb.processing["behavior"].data_interfaces.keys()))
    try:
        from nwbwidgets import nwb2widget
        display(nwb2widget(nwb))
    except Exception as exc:
        print("nwbwidgets could not render in this environment:", repr(exc))
        print("Falling back to the printed key inventory above.")


In [ ]:
session = load_session_arrays(selected.local_path)

In [ ]:
def plot_spike_raster(spike_times_by_unit, max_units=12, t_start=0.0, t_end=10.0, out_path=Path("artifacts/spike_raster.png")):
    """
    INPUT:
    spike_times_by_unit : one spike-time array per unit (List[np.ndarray])
    max_units           : maximum number of units to plot (int)
    t_start             : left edge of the plotting window in seconds (float)
    t_end               : right edge of the plotting window in seconds (float)
    out_path            : file path where the raster figure should be saved (Path)

    OUTPUT:
    None (displays and saves figure)
    """
    # pseudocode: create the raster figure and its plotting axis
    fig, ax = plt.subplots(figsize=(10, 4))
    # pseudocode: draw spike marks for the first few units within the requested time window
    for unit_idx, spike_times in enumerate(spike_times_by_unit[:max_units]):
        # pseudocode: keep only spikes that fall inside the displayed time range
        mask = (spike_times >= t_start) & (spike_times <= t_end)
        # pseudocode: draw one vertical tick mark per spike for this unit
        ax.vlines(spike_times[mask], unit_idx + 0.6, unit_idx + 1.4, linewidth=0.8)
    # pseudocode: title the raster and label its axes for readability
    ax.set_title(f"Spike raster for the first {max_units} units")
    ax.set_xlabel("Time (s)")
    ax.set_ylabel("Unit index")
    # pseudocode: tighten the layout, save the figure, show it, and then close it
    fig.tight_layout()
    fig.savefig(out_path, dpi=160)
    plt.show()
    plt.close(fig)

def plot_velocity_trace(timestamps, vel, t_start=0.0, t_end=10.0, out_path=Path("artifacts/cursor_velocity_trace.png")):
    """
    INPUT:
    timestamps : timestamps aligned to the velocity samples (n_vel,)
    vel        : two-dimensional cursor velocity trace (n_vel, 2)
    t_start    : left edge of the plotting window in seconds (float)
    t_end      : right edge of the plotting window in seconds (float)
    out_path   : file path where the trace figure should be saved (Path)

    OUTPUT:
    None (displays and saves figure)
    """
    # pseudocode: keep only the time samples that fall inside the requested plotting window
    mask = (timestamps >= t_start) & (timestamps <= t_end)
    # pseudocode: create one subplot for vx and one for vy
    fig, axes = plt.subplots(2, 1, figsize=(10, 4.5), sharex=True)
    # pseudocode: plot the x-velocity component over time
    axes[0].plot(timestamps[mask], vel[mask, 0])
    axes[0].set_ylabel("vx (cm/s)")
    # pseudocode: plot the y-velocity component over time
    axes[1].plot(timestamps[mask], vel[mask, 1])
    axes[1].set_ylabel("vy (cm/s)")
    axes[1].set_xlabel("Time (s)")
    # pseudocode: title the figure, save it, display it, and then close it
    fig.suptitle("Cursor velocity trace")
    fig.tight_layout()
    fig.savefig(out_path, dpi=160)
    plt.show()
    plt.close(fig)

# pseudocode: render and save a quick spike raster example from the loaded session
plot_spike_raster(session["spike_times_by_unit"])
# pseudocode: render and save a quick cursor-velocity trace from the loaded session
plot_velocity_trace(session["vel_timestamps"], session["vel_values"])

# Part 4: Preprocessing + Split Construction

In [ ]:
trial_df = choose_successful_trials(session["trials"], context_sec=CONTEXT_SEC)
split_map = split_trials(trial_df, RANDOM_SEED)
sample_df = build_sample_table(
    session["vel_timestamps"],
    session["vel_values"],
    trial_df,
    split_map,
    context_sec=CONTEXT_SEC,
    stride_sec=TARGET_STRIDE_SEC,
)

split_frames = {
    split: sample_df.loc[sample_df["split"] == split].copy().reset_index(drop=True)
    for split in ["train", "val", "test"]
}
if FAST_DEV_RUN:
    for split, limit in FAST_DEV_MAX_SAMPLES.items():
        split_frames[split] = split_frames[split].head(limit).copy().reset_index(drop=True)

split_summary = pd.DataFrame(
    {
        "num_trials": {split: len(split_map[split]) for split in split_map},
        "num_samples": {split: len(split_frames[split]) for split in split_frames},
        "trial_ids_head": {split: split_map[split][:5].tolist() for split in split_map},
    }
)
display(split_summary)

In [ ]:
end_time = max(
    session["vel_timestamps"][-1],
    max(st[-1] for st in session["spike_times_by_unit"] if len(st)),
)
binned_counts = bin_full_session(
    session["spike_times_by_unit"],
    end_time=end_time,
    bin_size_sec=BIN_SIZE_SEC,
)
context_bins = int(round(CONTEXT_SEC / BIN_SIZE_SEC))

In [ ]:
X_train, y_train = build_baseline_arrays(split_frames["train"], binned_counts, context_bins)
X_val, y_val = build_baseline_arrays(split_frames["val"], binned_counts, context_bins)
X_test, y_test = build_baseline_arrays(split_frames["test"], binned_counts, context_bins)

X_train_flat = X_train.reshape(len(X_train), -1)
X_val_flat = X_val.reshape(len(X_val), -1)
X_test_flat = X_test.reshape(len(X_test), -1)

feat_mean = X_train_flat.mean(axis=0, keepdims=True)
feat_std = X_train_flat.std(axis=0, keepdims=True) + 1e-6
X_train_flat_n = (X_train_flat - feat_mean) / feat_std
X_val_flat_n = (X_val_flat - feat_mean) / feat_std
X_test_flat_n = (X_test_flat - feat_mean) / feat_std

In [ ]:
X_train_seq_n = X_train_flat_n.reshape(len(X_train), context_bins, X_train.shape[-1])
X_val_seq_n = X_val_flat_n.reshape(len(X_val), context_bins, X_val.shape[-1])
X_test_seq_n = X_test_flat_n.reshape(len(X_test), context_bins, X_test.shape[-1])

y_mean = y_train.mean(axis=0, keepdims=True)
y_std = y_train.std(axis=0, keepdims=True) + 1e-6
y_train_n = ((y_train - y_mean) / y_std).astype(np.float32)
y_val_n = ((y_val - y_mean) / y_std).astype(np.float32)
y_test_n = ((y_test - y_mean) / y_std).astype(np.float32)

print("Feature tensor shapes:")
print("X_train:", X_train.shape, "y_train:", y_train.shape)
print("X_val:", X_val.shape, "y_val:", y_val.shape)
print("X_test:", X_test.shape, "y_test:", y_test.shape)

# Part 5: Baseline 1 - Wiener Filter

The Wiener filter baseline is a linear decoder: flatten the last `1s` of binned spike counts and learn a linear mapping to the current 2D cursor velocity. This is a classic neural decoding baseline because it is fast, interpretable, and provides a good sanity check before training nonlinear models.

In [ ]:
results = {}

ridge = Ridge(alpha=1.0)
ridge.fit(X_train_flat_n, y_train)
wiener_pred = ridge.predict(X_test_flat_n).astype(np.float32)
results["Wiener"] = r2_metrics(y_test, wiener_pred)

plot_pred_vs_actual(split_frames["test"], wiener_pred, "Wiener", ARTIFACTS_DIR / "wiener_pred_vs_actual.png")
display(pd.DataFrame([results["Wiener"]], index=["Wiener"]))


# Part 6: Baseline 2 - MLP

The MLP baseline uses the same `1s` spike-count history as the Wiener filter, but replaces the linear readout with a small multilayer perceptron. This lets us test whether a simple nonlinear function of the same binned features already closes part of the gap to POYO.

In [ ]:
mlp = MLPDecoder(input_dim=X_train_flat_n.shape[1]).to(DEVICE)
mlp_train_loader = make_tensor_loader(X_train_flat_n.astype(np.float32), y_train_n, MLP_BATCH_SIZE, True)
mlp_val_loader = make_tensor_loader(X_val_flat_n.astype(np.float32), y_val_n, MLP_BATCH_SIZE, False)
mlp_test_loader = make_tensor_loader(X_test_flat_n.astype(np.float32), y_test_n, MLP_BATCH_SIZE, False)

mlp, mlp_hist = train_torch_decoder(
    mlp,
    mlp_train_loader,
    mlp_val_loader,
    MAX_EPOCHS_MLP,
    lr=3e-4,
    target_mean=y_mean,
    target_std=y_std,
)

In [ ]:
mlp_test_metrics, mlp_pred, _ = iterate_batches(
    mlp_test_loader,
    mlp,
    optimizer=None,
    target_mean=y_mean,
    target_std=y_std,
)
results["MLP"] = {k: float(v) for k, v in mlp_test_metrics.items() if k.startswith("r2")}

plot_learning_curve(mlp_hist, "MLP learning curve", ARTIFACTS_DIR / "mlp_learning_curve.png")
plot_pred_vs_actual(split_frames["test"], mlp_pred, "MLP", ARTIFACTS_DIR / "mlp_pred_vs_actual.png")
display(pd.DataFrame([results["MLP"]], index=["MLP"]))

# Part 7: Baseline 3 - GRU

The GRU baseline keeps the `1s` history as an explicit sequence instead of flattening it. This is a closer sequential baseline for POYO because the model can accumulate information over the spike-count bins instead of seeing them only as a static vector.

In [ ]:
gru = GRUDecoder(input_dim=X_train_seq_n.shape[-1]).to(DEVICE)
gru_train_loader = make_tensor_loader(X_train_seq_n.astype(np.float32), y_train_n, GRU_BATCH_SIZE, True)
gru_val_loader = make_tensor_loader(X_val_seq_n.astype(np.float32), y_val_n, GRU_BATCH_SIZE, False)
gru_test_loader = make_tensor_loader(X_test_seq_n.astype(np.float32), y_test_n, GRU_BATCH_SIZE, False)

gru, gru_hist = train_torch_decoder(
    gru,
    gru_train_loader,
    gru_val_loader,
    MAX_EPOCHS_GRU,
    lr=3e-4,
    target_mean=y_mean,
    target_std=y_std,
)

In [ ]:
gru_test_metrics, gru_pred, _ = iterate_batches(
    gru_test_loader,
    gru,
    optimizer=None,
    target_mean=y_mean,
    target_std=y_std,
)
results["GRU"] = {k: float(v) for k, v in gru_test_metrics.items() if k.startswith("r2")}

plot_learning_curve(gru_hist, "GRU learning curve", ARTIFACTS_DIR / "gru_learning_curve.png")
plot_pred_vs_actual(split_frames["test"], gru_pred, "GRU", ARTIFACTS_DIR / "gru_pred_vs_actual.png")
display(pd.DataFrame([results["GRU"]], index=["GRU"]))

# Part 8: POYO Single-Session Training Using the Official Implementation

POYO differs from the baselines in two important ways:

1. It treats **individual spikes as tokens** instead of first binning them into firing rates.
2. It uses a **Perceiver-style latent bottleneck** with time-aware attention, so the model can compress a variable number of spikes in a `1s` context window before decoding velocity.

Below, the educational diagram is generated directly in Python so the notebook stays self-contained.


In [ ]:
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch

def draw_poyo_diagram(out_path=Path("artifacts/poyo_architecture_diagram.png")):
    """
    INPUT:
    out_path : file path for saved diagram (Path)

    OUTPUT:
    None (displays and saves figure)
    """
    # pseudocode: create a wide figure and axis for the horizontal POYO architecture diagram
    fig, ax = plt.subplots(figsize=(11, 3.8))
    # pseudocode: fix the axis limits so every diagram element uses normalized coordinates
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    # pseudocode: hide the axes because this is a conceptual diagram rather than a data plot
    ax.axis("off")

    def box(x, y, w, h, text, color):
        """
        INPUT:
        x     : left coordinate of the box (float)
        y     : bottom coordinate of the box (float)
        w     : box width (float)
        h     : box height (float)
        text  : label drawn in the center of the box (str)
        color : fill color for the box (str)

        OUTPUT:
        None (adds rounded box to axes)
        """
        # pseudocode: create a rounded rectangle patch for one conceptual POYO stage
        patch = FancyBboxPatch(
            (x, y), w, h,
            boxstyle="round,pad=0.02,rounding_size=0.02",
            linewidth=1.5,
            facecolor=color,
            edgecolor="black",
        )
        # pseudocode: add the rounded rectangle to the diagram axis
        ax.add_patch(patch)
        # pseudocode: place the stage label at the center of the box
        ax.text(x + w / 2, y + h / 2, text, ha="center", va="center", fontsize=11)

    def arrow(x0, y0, x1, y1):
        """
        INPUT:
        x0 : x-coordinate of the arrow tail (float)
        y0 : y-coordinate of the arrow tail (float)
        x1 : x-coordinate of the arrow head (float)
        y1 : y-coordinate of the arrow head (float)

        OUTPUT:
        None (adds arrow to axes)
        """
        # pseudocode: draw a directed arrow showing information flow between POYO stages
        ax.add_patch(FancyArrowPatch((x0, y0), (x1, y1), arrowstyle="->", mutation_scale=12, linewidth=1.5))

    # pseudocode: draw the spike-token input stage at the far left
    box(0.03, 0.22, 0.18, 0.56, "Spike tokens\n(unit id + timestamp)", "#d8ecff")
    # pseudocode: draw the cross-attention stage that writes spike information into latents
    box(0.29, 0.22, 0.18, 0.56, "Cross-attention\ninto latent tokens", "#dff5dd")
    # pseudocode: draw the latent self-attention processing stack in the middle
    box(0.55, 0.22, 0.18, 0.56, "Latent self-attention\nprocessing stack", "#fff0c9")
    # pseudocode: draw the query-based readout stage that predicts velocity at the output time
    box(0.81, 0.22, 0.16, 0.56, "Query at output time\n-> velocity", "#ffd9d9")
    # pseudocode: connect spike tokens to the cross-attention block
    arrow(0.21, 0.50, 0.29, 0.50)
    # pseudocode: connect cross-attention outputs to the latent processing stack
    arrow(0.47, 0.50, 0.55, 0.50)
    # pseudocode: connect latent processing to the final query readout
    arrow(0.73, 0.50, 0.81, 0.50)
    # pseudocode: title the diagram with a one-sentence summary of POYO's information flow
    ax.set_title("POYO in one sentence: spikes become tokens, latents summarize them, queries read out velocity")
    # pseudocode: tighten the layout, save the figure, display it, and then close it
    fig.tight_layout()
    fig.savefig(out_path, dpi=180)
    plt.show()
    plt.close(fig)

# pseudocode: render and save the POYO architecture diagram
draw_poyo_diagram()


In [ ]:
readout_spec = MODALITY_REGISTRY["cursor_velocity_2d"]
poyo = POYO(
    sequence_length=1.0,
    latent_step=0.125,
    num_latents_per_step=16,
    dim=64,
    depth=6,
    dim_head=64,
    cross_heads=2,
    self_heads=8,
    ffn_dropout=0.2,
    lin_dropout=0.4,
    atn_dropout=0.2,
    readout_spec=readout_spec,
)

session_id = Path(selected.path).stem.replace("_behavior+ecephys", "")
poyo.unit_emb.initialize_vocab(session["unit_ids"])
poyo.session_emb.initialize_vocab([session_id])

In [ ]:
train_ds = POYOSingleQueryDataset(
    split_frames["train"],
    session["spike_times_by_unit"],
    session["unit_ids"],
    session_id,
    poyo,
)
val_ds = POYOSingleQueryDataset(
    split_frames["val"],
    session["spike_times_by_unit"],
    session["unit_ids"],
    session_id,
    poyo,
)
test_ds = POYOSingleQueryDataset(
    split_frames["test"],
    session["spike_times_by_unit"],
    session["unit_ids"],
    session_id,
    poyo,
)

train_loader = DataLoader(train_ds, batch_size=POYO_BATCH_SIZE, shuffle=True, collate_fn=tb_collate)
val_loader = DataLoader(val_ds, batch_size=POYO_BATCH_SIZE, shuffle=False, collate_fn=tb_collate)
test_loader = DataLoader(test_ds, batch_size=POYO_BATCH_SIZE, shuffle=False, collate_fn=tb_collate)

In [ ]:
poyo, poyo_hist = train_poyo_model(poyo, train_loader, val_loader, MAX_EPOCHS_POYO)
poyo_test_metrics, poyo_pred, _ = evaluate_poyo_loader(poyo, test_loader, denorm_std=20.0)
results["POYO"] = {k: float(v) for k, v in poyo_test_metrics.items() if k.startswith("r2")}

plot_learning_curve(poyo_hist, "POYO learning curve", ARTIFACTS_DIR / "poyo_learning_curve.png")
plot_pred_vs_actual(split_frames["test"], poyo_pred, "POYO", ARTIFACTS_DIR / "poyo_pred_vs_actual.png")
display(pd.DataFrame([results["POYO"]], index=["POYO"]))

# Part 9: Evaluation + Comparisons

This section aggregates all four models into a single comparison table and saves the final metrics under `./artifacts/metrics.json`. The paper's Section 3.2 reports an average single-session `R^2 ≈ 0.84` on random-target sessions and a higher average on center-out sessions, so the fairest paper reference for our auto-selected asset is the random-target regime.

In [ ]:
comparison_df = pd.DataFrame(results).T.loc[["Wiener", "MLP", "GRU", "POYO"]]
display(comparison_df)

fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(comparison_df.index, comparison_df["r2_mean"], color=["#8db1ff", "#ffbe7a", "#7ecf9a", "#ff8f8f"])
ax.axhline(0.8402, color="black", linestyle="--", linewidth=1.2, label="POYO paper RT average (Section 3.2)")
ax.set_ylabel("Test R^2")
ax.set_title("Model comparison on the held-out split")
ax.legend(loc="lower right")
fig.tight_layout()
fig.savefig(ARTIFACTS_DIR / "comparison_bar_chart.png", dpi=180)
plt.show()
plt.close(fig)

In [ ]:
run_metadata = {
    "fast_dev_run": FAST_DEV_RUN,
    "random_seed": RANDOM_SEED,
    "context_sec": CONTEXT_SEC,
    "bin_size_sec": BIN_SIZE_SEC,
    "target_stride_sec": TARGET_STRIDE_SEC,
    "max_epochs": {
        "mlp": MAX_EPOCHS_MLP,
        "gru": MAX_EPOCHS_GRU,
        "poyo": MAX_EPOCHS_POYO,
    },
    "device": DEVICE,
    "torch_brain_clone_commit": clone_commit,
    "torch_brain_version": getattr(torch_brain, "__version__", "unknown"),
    "selected_asset": selected.__dict__,
    "nwb_structure": structure,
    "sample_counts": {split: int(len(df)) for split, df in split_frames.items()},
}

with open(ARTIFACTS_DIR / "metrics.json", "w") as f:
    json.dump(
        {
            "results": results,
            "selected_asset": selected.__dict__,
            "sample_counts": {split: int(len(df)) for split, df in split_frames.items()},
            "paper_reference": {
                "proposal_target": "~0.97 on an easier center-out setting",
                "paper_single_session_rt_average_r2": 0.8402,
            },
        },
        f,
        indent=2,
        default=json_default,
    )

In [ ]:
with open(ARTIFACTS_DIR / "run_metadata.json", "w") as f:
    json.dump(run_metadata, f, indent=2, default=json_default)

print("Saved metrics to:", ARTIFACTS_DIR / "metrics.json")
print("Saved run metadata to:", ARTIFACTS_DIR / "run_metadata.json")

# Part 10: Repro Notes

In [ ]:
repro_df = pd.DataFrame(
    [
        {
            "device": DEVICE,
            "fast_dev_run": FAST_DEV_RUN,
            "random_seed": RANDOM_SEED,
            "context_sec": CONTEXT_SEC,
            "bin_size_sec": BIN_SIZE_SEC,
            "target_stride_sec": TARGET_STRIDE_SEC,
            "clone_commit": clone_commit,
            "asset_path": selected.path,
            "asset_size_mb": round(selected.size_bytes / 1e6, 3),
            "note": "Best-effort single-session reproduction on the smallest valid published session.",
        }
    ]
)
display(repro_df)

print("Known limitations:")
print("- The notebook auto-selects the smallest valid session, which may not match the exact session distribution used in the paper averages.")
print("- FAST_DEV_RUN trades off accuracy for speed; switch it off for a stronger best-effort Colab run.")
print("- The paper uses richer weighting and evaluation intervals for the Perich/Miller datasets; here we keep the split faithful and the runtime practical for a single Colab session.")
